# Rolling Window Sensor Feature Engineering

Raw sensor readings are noisy and don't capture trends over time. This notebook computes rolling mean, standard deviation, and variance over a 5-row window for each sensor signal, giving the model short-term trend and volatility information that a single reading cannot provide.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/raw/ai4i2020.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
sensor_cols = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

In [ ]:
window = 5

for col in sensor_cols:
    df[f'{col}_roll_mean'] = df[col].rolling(window=window, min_periods=1).mean()
    df[f'{col}_roll_std']  = df[col].rolling(window=window, min_periods=1).std().fillna(0)
    df[f'{col}_roll_var']  = df[col].rolling(window=window, min_periods=1).var().fillna(0)

print("New shape:", df.shape)
print("New columns added:", [c for c in df.columns if 'roll' in c])

## Verifying rolling features visually

Comparing the raw signal against its rolling mean confirms the smoothing behavior is working as expected before we use these features downstream.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['Rotational speed [rpm]'][:100], label='Original')
plt.plot(df['Rotational speed [rpm]_roll_mean'][:100], label='Rolling Mean')
plt.title('Rotational Speed - Original vs Rolling Mean')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df.to_csv("../data/processed/ai4i2020_features.csv", index=False)
print("Saved! Total features:", df.shape[1])

## Summary

Rolling features increased the feature count from 14 to 29 columns (15 new rolling statistics across 5 sensors). This enriched dataset is saved to `data/processed/ai4i2020_features.csv` for use in the data fusion step.